# C0

In [7]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
from PIL import Image

print("Rechargement des données et création des DataLoaders...")

# 1. Chargement des CSV propres (sauvegardés à la Phase 2)
train_df = pd.read_csv('../data/processed/tabular_train.csv')
val_df = pd.read_csv('../data/processed/tabular_val.csv')

# 2. Recréation des Pipelines de Transformation
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_transforms = transforms.Compose([
    transforms.Resize(232), transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20), transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])
val_transforms = transforms.Compose([
    transforms.Resize(232), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# 3. Recréation de la classe Dataset
class HAM10000Dataset(Dataset):
    def __init__(self, dataframe, transform=None, label_map=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.label_map = label_map
    def __len__(self): return len(self.dataframe)
    def __getitem__(self, idx):
        img_path = self.dataframe.loc[idx, 'path']
        image = Image.open(img_path).convert('RGB')
        label_str = self.dataframe.loc[idx, 'dx']
        if self.transform: image = self.transform(image)
        return image, torch.tensor(self.label_map[label_str], dtype=torch.long)

# 4. Configuration du Sampler et des DataLoaders
classes = sorted(train_df['dx'].unique())
label_map = {class_name: idx for idx, class_name in enumerate(classes)}

class_counts = train_df['dx'].value_counts().to_dict()
sample_weights = [(1.0 / class_counts[label]) for label in train_df['dx']]
sampler = WeightedRandomSampler(torch.DoubleTensor(sample_weights), len(sample_weights), replacement=True)

BATCH_SIZE = 32
train_dataset = HAM10000Dataset(train_df, transform=train_transforms, label_map=label_map)
val_dataset = HAM10000Dataset(val_df, transform=val_transforms, label_map=label_map)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("✅ Contexte rechargé : 'train_loader' et 'val_loader' sont de nouveau disponibles !")

Rechargement des données et création des DataLoaders...
✅ Contexte rechargé : 'train_loader' et 'val_loader' sont de nouveau disponibles !


# Phase 3 : Modélisation (CNN from scratch)

**Objectif :** Construire, entraîner et évaluer un Réseau de Neurones Convolutif (CNN) conçu de zéro, sans utiliser de modèle pré-entraîné. Cela démontre une compréhension profonde de l'extraction de caractéristiques spatiales.

## 1. Architecture du Réseau (La classe `CustomCNN`)

Notre réseau prendra en entrée des images au format `[Batch, 3, 224, 224]`.
Il est composé de 4 blocs convolutifs. Chaque bloc contient :
* Une **Convolution** (`Conv2d`) avec un padding de 1 pour ne pas rogner les bords.
* Une **Normalisation par lot** (`BatchNorm2d`) pour stabiliser et accélérer l'apprentissage.
* Une fonction d'activation **ReLU** pour introduire de la non-linéarité.
* Un **Max Pooling** (`MaxPool2d`) qui divise la taille spatiale de l'image par 2.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print("Définition de l'architecture CustomCNN...")

class CustomCNN(nn.Module):
    def __init__(self, num_classes=7):
        super(CustomCNN, self).__init__()
        
        # ==========================================
        # 1. EXTRACTEUR DE CARACTÉRISTIQUES (Feature Extractor)
        # Entrée : Image [3, 224, 224]
        # ==========================================
        
        # Bloc 1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) 
        # Sortie Bloc 1 : [16, 112, 112]

        # Bloc 2
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        # Sortie Bloc 2 : [32, 56, 56]

        # Bloc 3
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        # Sortie Bloc 3 : [64, 28, 28]

        # Bloc 4
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(128)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)
        # Sortie Bloc 4 : [128, 14, 14]

        # ==========================================
        # 2. CLASSIFIEUR (Dense Layers)
        # ==========================================
        
        # Calcul de la taille du tenseur aplati (Flatten) :
        # 128 canaux * 14 hauteur * 14 largeur = 25088
        self.flatten_size = 128 * 14 * 14
        
        self.fc1 = nn.Linear(self.flatten_size, 512)
        self.dropout = nn.Dropout(0.5) # Désactive 50% des neurones (Anti-Overfitting)
        self.fc2 = nn.Linear(512, num_classes) # Sortie finale : 7 neurones

    def forward(self, x):

        '''
            conv1(x) → applique 16 filtres → extrait des features
            bn1(...) → normalise
            F.relu(...) → ajoute de la non-linéarité (met les négatifs à 0)
            pool1(...) → réduit la taille
        '''

        # Passage dans l'extracteur
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = self.pool4(F.relu(self.bn4(self.conv4(x))))
        
        # Aplatissement (Flatten) : On passe de 3D à 1D
        # x.size(0) garde la taille du batch intacte
        x = x.view(x.size(0), -1) 
        
        # Passage dans le classifieur
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

print("✅ Classe CustomCNN compilée avec succès !")

Définition de l'architecture CustomCNN...
✅ Classe CustomCNN compilée avec succès !


### 1.1 Le "Sanity Check" du réseau (Vérification des dimensions)

Une erreur classique en PyTorch est de se tromper dans le calcul de la taille de la couche `Linear` après les convolutions. Pour vérifier que notre mathématique ($128 \times 14 \times 14$) est correcte, nous allons faire passer une "fausse image" aléatoire à travers le réseau sans l'entraîner.

In [3]:
# Instanciation du modèle
model = CustomCNN(num_classes=7)

# Création d'un tenseur aléatoire simulant 1 image (Batch=1, Canaux=3, H=224, W=224)
dummy_input = torch.randn(1, 3, 224, 224)

# Passage de l'image dans le modèle (Forward Pass)
try:
    dummy_output = model(dummy_input)
    print("✅ TEST RÉUSSI : L'architecture est mathématiquement valide !")
    print(f"Forme de l'entrée : {dummy_input.shape}")
    print(f"Forme de la sortie : {dummy_output.shape} -> (Batch Size, Nombre de Classes)")
    
    # Affichage du nombre total de paramètres (poids) du modèle
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Nombre total de paramètres à entraîner : {total_params:,}")
    
except Exception as e:
    print("❌ ERREUR DE DIMENSION :")
    print(e)

✅ TEST RÉUSSI : L'architecture est mathématiquement valide !
Forme de l'entrée : torch.Size([1, 3, 224, 224])
Forme de la sortie : torch.Size([1, 7]) -> (Batch Size, Nombre de Classes)
Nombre total de paramètres à entraîner : 12,947,079


## 2. Configuration du Moteur d'Apprentissage

Pour entraîner le réseau, nous définissons l'environnement d'exécution (CPU/GPU) et les hyperparamètres d'optimisation.
* **Device :** Détection automatique du matériel.
* **Fonction de perte (Criterion) :** `CrossEntropyLoss`. (Note cruciale : nous n'appliquons **aucune pondération de classe** ici, car le déséquilibre est déjà géré en amont par le `WeightedRandomSampler` dans notre `train_loader`).
* **Optimiseur :** `Adam` (Adaptive Moment Estimation), particulièrement efficace pour les modèles entraînés de zéro (from scratch) grâce à son taux d'apprentissage adaptatif.

In [4]:
import torch.optim as optim

print("Configuration du moteur d'entraînement...")

# 1. Sélection automatique du matériel (Device)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Matériel détecté pour l'entraînement : {device.type.upper()}")

# 2. Instanciation finale du modèle et transfert vers le matériel (CPU ou GPU)
model = CustomCNN(num_classes=7).to(device)

# 3. Définition de la fonction de perte (Loss)
# On utilise la version standard car le DataLoader s'occupe de l'équilibrage
criterion = nn.CrossEntropyLoss()

# 4. Définition de l'Optimiseur
# Le Learning Rate (lr) est fixé à 0.001 (1e-3), un standard robuste pour Adam
LEARNING_RATE = 0.001
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("✅ Moteur d'apprentissage configuré avec succès !")

Configuration du moteur d'entraînement...
Matériel détecté pour l'entraînement : CPU
✅ Moteur d'apprentissage configuré avec succès !


## 3. Création des Boucles d'Entraînement et de Validation

En PyTorch, l'entraînement nécessite de coder manuellement la boucle d'apprentissage.
Nous allons définir deux fonctions distinctes :
1. **`train_step` :** Active le calcul du gradient, effectue la propagation avant (*forward*), calcule l'erreur, et met à jour les poids (*backward*).
2. **`val_step` :** Désactive le calcul du gradient (`torch.no_grad()`) pour économiser la mémoire et évalue les performances du modèle sur des images qu'il n'a jamais vues.

**Contrainte Matérielle :** Étant exécuté sur un processeur (CPU), nous intégrons un `DEBUG_MODE`. S'il est activé, l'entraînement s'arrêtera artificiellement après 2 lots (batches) afin de valider le bon fonctionnement du code sans surcharger la machine.

In [5]:
print("Définition des fonctions d'entraînement et de validation...")

def train_step(model, dataloader, criterion, optimizer, device, debug=False):
    model.train() # Passe le modèle en mode entraînement (active le Dropout et la BatchNorm)
    running_loss = 0.0
    
    for batch_idx, (inputs, labels) in enumerate(dataloader):
        # Envoi des données sur le CPU/GPU
        inputs, labels = inputs.to(device), labels.to(device)
        
        # 1. Remise à zéro des gradients
        optimizer.zero_grad()
        
        # 2. Propagation avant (Forward pass)
        outputs = model(inputs)
        
        # 3. Calcul de l'erreur (Loss)
        loss = criterion(outputs, labels)
        
        # 4. Rétropropagation (Backward pass)
        loss.backward()
        
        # 5. Mise à jour des poids
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        
        # --- MODE DÉBOGAGE POUR CPU ---
        if debug and batch_idx >= 1: # S'arrête après 2 batches (0 et 1)
            print("   [Debug] Interruption de l'entraînement après 2 batches.")
            break
            
    # Calcul de la perte moyenne pour cette époque
    # Si on est en debug, on divise seulement par le nombre d'images vues (2 batches * 32 = 64)
    processed_samples = 64 if debug else len(dataloader.dataset)
    epoch_loss = running_loss / processed_samples
    return epoch_loss

def val_step(model, dataloader, criterion, device, debug=False):
    model.eval() # Passe le modèle en mode évaluation (désactive Dropout)
    running_loss = 0.0
    correct_predictions = 0
    total_predictions = 0
    
    # torch.no_grad() désactive le graphe de calcul = Moins de RAM, calculs plus rapides
    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(dataloader):
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            
            # Calcul de la précision (Accuracy)
            _, predicted = torch.max(outputs.data, 1) # On prend la classe avec la plus forte probabilité
            total_predictions += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()
            
            # --- MODE DÉBOGAGE POUR CPU ---
            if debug and batch_idx >= 1:
                print("   [Debug] Interruption de la validation après 2 batches.")
                break
                
    processed_samples = 64 if debug else len(dataloader.dataset)
    epoch_loss = running_loss / processed_samples
    
    # En mode debug, total_predictions vaudra 64. Sinon, la taille du dataset de validation.
    epoch_acc = (correct_predictions / total_predictions) * 100 
    return epoch_loss, epoch_acc

print("✅ Fonctions 'train_step' et 'val_step' créées !")

Définition des fonctions d'entraînement et de validation...
✅ Fonctions 'train_step' et 'val_step' créées !


### 3.1 La Boucle Principale (Epochs) et le Checkpointing

Nous allons maintenant exécuter notre modèle sur plusieurs "Époques" (passages complets sur le dataset). 
À chaque époque, nous vérifierons si l'erreur de validation (Validation Loss) a diminué. Si c'est le cas, cela signifie que notre modèle s'améliore, et nous sauvegarderons ses poids mathématiques sur le disque dur (`.pth`). C'est ce qu'on appelle le **Checkpointing**.

In [8]:
import os
import time

# Création du dossier pour sauvegarder les modèles
os.makedirs('../models', exist_ok=True)

# ==========================================
# PARAMÈTRES DE LA BOUCLE
# ==========================================
NUM_EPOCHS = 3 # On fait 3 passages pour le test
DEBUG_MODE = True # ⚠️ VITAL POUR LE TEST LOCAL SUR CPU ⚠️

print(f"Lancement de l'entraînement pour {NUM_EPOCHS} époques. (Debug Mode: {DEBUG_MODE})\n")

# Historique pour tracer les courbes plus tard
history = {
    'train_loss': [],
    'val_loss': [],
    'val_acc': []
}

best_val_loss = float('inf') # On initialise la meilleure erreur à l'infini

# La boucle principale
for epoch in range(NUM_EPOCHS):
    start_time = time.time()
    
    print(f"--- Époque {epoch+1}/{NUM_EPOCHS} ---")
    
    # 1. Entraînement
    train_loss = train_step(model, train_loader, criterion, optimizer, device, debug=DEBUG_MODE)
    
    # 2. Validation
    val_loss, val_acc = val_step(model, val_loader, criterion, device, debug=DEBUG_MODE)
    
    # 3. Sauvegarde de l'historique
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # 4. Checkpointing (Sauvegarde du meilleur modèle)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '../models/custom_cnn_best.pth')
        save_msg = "🌟 Nouveau meilleur modèle sauvegardé !"
    else:
        save_msg = ""
        
    end_time = time.time()
    epoch_duration = end_time - start_time
    
    # Affichage des résultats de l'époque
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Accuracy: {val_acc:.2f}%")
    print(f"Temps écoulé: {epoch_duration:.1f} secondes. {save_msg}\n")

print("✅ Entraînement terminé !")

Lancement de l'entraînement pour 3 époques. (Debug Mode: True)

--- Époque 1/3 ---
   [Debug] Interruption de l'entraînement après 2 batches.
   [Debug] Interruption de la validation après 2 batches.
Train Loss: 8.8368 | Val Loss: 2.2836 | Val Accuracy: 0.00%
Temps écoulé: 9.5 secondes. 🌟 Nouveau meilleur modèle sauvegardé !

--- Époque 2/3 ---
   [Debug] Interruption de l'entraînement après 2 batches.
   [Debug] Interruption de la validation après 2 batches.
Train Loss: 16.3140 | Val Loss: 2.0403 | Val Accuracy: 0.00%
Temps écoulé: 5.9 secondes. 🌟 Nouveau meilleur modèle sauvegardé !

--- Époque 3/3 ---
   [Debug] Interruption de l'entraînement après 2 batches.
   [Debug] Interruption de la validation après 2 batches.
Train Loss: 10.1545 | Val Loss: 0.5591 | Val Accuracy: 100.00%
Temps écoulé: 5.5 secondes. 🌟 Nouveau meilleur modèle sauvegardé !

✅ Entraînement terminé !
